In [1]:
import os
import sys
import re
import glob
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv()
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNINGS'] = '1'
os.environ['TOKENIZER_PARALLELISM'] = 'false'

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb
import faiss
import pypdf

c:\Users\Windows\Documents\CloudDesko\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATASET_DIR = "clouddesk_dataset"

source_summary = []
for folder in os.listdir(DATASET_DIR):
    folder_path = os.path.join(DATASET_DIR, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        source_summary.append({
            'folder': folder,
            'file count': len(files),
            'sample files': ','.join(files[:2])
        })

df_audit = pd.DataFrame(source_summary)
try: display(df_audit)
except NameError: print(df_audit)


,folder,file count,sample files
0,api_documentation,4,"authentication.md,rest-api-endpoints.md"
1,engineering_runbooks,3,"runbook-api-latency-incident.md,runbook-sso-sa..."
2,help_center_articles,5,"HC-101_integrating-clouddesk-with-slack.pdf,HC..."
3,release_notes,3,"RN-3.4.0_release-notes.pdf,RN-3.5.0_release-no..."
4,support_tickets,1,previous_support_tickets.csv


In [5]:
class DocumentChunk:
    def __init__(self, text: str, metadata: dict):
        self.page_content = text.strip()
        self.metadata = metadata
    def __repr__(self):
        return f"<chunk id={self.metadata.get('chunk_id')} type={self.metadata.get('source_type')} title={self.metadata.get('title')}>"


def parse_markdown(filepath: str, source_type: str) -> tuple:
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    meta = {
        'source_type': source_type,
        #'title': os.path.splitext(os.path.basename(filepath))[0].replace('-', '').title(),
        'title': os.path.splitext(os.path.basename(filepath))[0].replace('-', ' ').title(),
        'last_updated': '2026-09-18',
        'version': 'v3.6.0',
        'source_file': os.path.basename(filepath)
    }
    body = content
    if content.startswith('---'):
        parts = content.split('---', 2)
        if len(parts) >= 3:
            body = parts[2]
            for line in parts[1].strip().split('\n'):
                if ':'in line:
                    k, v = line.split(':', 1)
                    meta[k.strip().lower()] = v.strip('"\'')
    for line in body.split('\n'):
        if line.startswith('#'):
            #meta['title']  = line.replace('#', '').strip()
            meta['title']  = line.replace('#', '').strip().strip('"').strip("'").strip()
            break
    return meta, body  

def parse_pdf(filepath: str, source_type: str) -> tuple:
    reader = pypdf.PdfReader(filepath)
    text = ''
    for p in reader.pages:
        text += (p.extract_text() or '') + '\n\n'

    meta = {
        'source_type': source_type,
        'title': os.path.splitext(os.path.basename(filepath))[0].replace('_', '').title(),
        'last_updated': '2026-09-18',
        'version': 'v3.6.0',
        'source_file': os.path.basename(filepath)
    }
    
    for line in text.split('\n')[:10]:
        if 'Version' in line:
            v_m = re.search(r'Version:\s*([\w\.]+)', line)
            if v_m: meta['version'] = v_m.group(1)

        if 'Last Updated' in line:
                    d_m = re.search(r'Last Updated:\s*([\w\.]+)', line)
                    if d_m: meta['last_updated'] = d_m.group(1)

    return meta, text 

chunks = []
counter = 0

for fp in glob.glob(os.path.join(DATASET_DIR,  'api_documentation', '*.md')):
    meta, body = parse_markdown(fp, 'api_documentation')
    for sec in body.split('\n##'):
        counter += 1
        m = meta.copy()
        m['chunk_id'] = f'api_doc_{counter:04d}'
        chunks.append(DocumentChunk(sec, m))





for fp in glob.glob(os.path.join(DATASET_DIR,  'engineering_runbooks', '*.md')):
    meta, body = parse_markdown(fp, 'engineering_runbooks')
    for sec in body.split('\n##'):
        if len(sec.strip()) > 30:
            counter += 1
            m = meta.copy()
            m['chunk_id'] = f'runbook_{counter:04d}'
            chunks.append(DocumentChunk(sec, m))


for fp in glob.glob(os.path.join(DATASET_DIR,  'help_center_articles', '*.pdf')):
    meta, text = parse_pdf(fp,'help_center_articles')
    for p in text.split('\n\n'):
        if len(p.strip()) > 40:
                counter += 1
                m = meta.copy()
                m['chunk_id'] = f'hc_art_{counter:04d}'
                chunks.append(DocumentChunk(p, m)) 
                   



for fp in glob.glob(os.path.join(DATASET_DIR,  'release_notes', '*.pdf')):
    meta, text = parse_pdf(fp,'release_notes')
    for p in text.split('\n\n'):
        if len(p.strip()) > 40:
                counter += 1
                m = meta.copy()
                m['chunk_id'] = f'rel_note_{counter:04d}'
                chunks.append(DocumentChunk(p, m))  


csv_p = os.path.join(DATASET_DIR, 'support_tickets', 'previous_support_tickets.csv')
df_t = pd.read_csv(csv_p)
for _, r in  df_t.iterrows():
    counter += 1
    tid = str(r['ticket_id'])
    content = f"Ticket [{tid}] subject: {r['subject']}\nQuestion: {r['customer_question']}\nResolution: {r['agent_answer']}"
    m = {
         'source_type': 'support_ticket',
         'title': f"Ticket {tid}",
         'last_updated': str(r['created_at']),
         'version': 'v3.6.0',
         'source_file': 'previous_support_tickets.csv',
         'source_type_reference': str(r.get('source_type_reference', '')),
         'source_title_reference': str(r.get('source_title_reference', '')),
        'chunk_id': f'ticket_{counter:04d}',   
         

    }

    chunks.append(DocumentChunk(content, m))                 


In [6]:
len(chunks)

110

In [7]:
from collections import Counter

c = Counter(ch.metadata.get('source_type') for ch in chunks)
for k, v in sorted(c.items()):
    print(f"{k}: {v}")
print("TOTAL chunks:", len(chunks))
print("Ticket rows in CSV:", len(df_t))

 api_documentation: 32
 engineering_runbook: 27
help_center_articles: 5
release_notes: 3
support_ticket: 43
TOTAL chunks: 110
Ticket rows in CSV: 43


In [8]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c.page_content for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)


print('Embedding matrix shape', embeddings.shape)
print(f'vector dimension: {embeddings.shape[1]}')

Batches: 100%|██████████| 4/4 [00:08<00:00,  2.22s/it]

Embedding matrix shape (110, 384)
vector dimension: 384


In [9]:
VECTOR_DB_DIR = 'vector_db'
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

# 1. Index into ChromeDB
chroma_client = chromadb.PersistentClient(path=os.path.join(VECTOR_DB_DIR, 'chroma'))

collection = chroma_client.get_or_create_collection(
    name='clouddesk_support_kb'

)


ids = [c.metadata['chunk_id'] for c in chunks]
documents = [c.page_content for c in chunks]
metadata = [c.metadata for c in chunks]

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadata


)

print(f'chromaDB Indexed: {collection.count()} vectors')

chromaDB Indexed: 110 vectors


In [10]:
#FAISS

In [11]:
pinecone_key = os.environ.get('PINECONE_API_KEY')
if pinecone_key:
    try:
        from pinecone import Pinecone, ServerlessSpec

        #pc = Pinecone(api_key=PINECONE_API_KEY)
        pc = Pinecone(api_key=pinecone_key)   

    
        index_name = os.environ.get('PINECONE_INDEX_NAME', 'clouddesko-support-rag')                                                     
        existing =  [idx.name for idx in pc.list_indexes()]
        if index_name not in existing:
            pc.create_index(name=index_name, dimension=384, metric='cosine',
                        spec=ServerlessSpec(cloud='aws', region='us-east-1') ) 
        pc_index= pc.Index(index_name)
        vectors_to_upsert = []
        for c, emb in zip(chunks, embeddings):
            meta = {k: str(v) for k, v, in c.metadata.items()}
            meta['page_content'] = c.page_content[:500]
            vectors_to_upsert.append((c.metadata['chunk_id'], emb.tolist(),
            meta ) )
        for i in range (0, len(vectors_to_upsert), 50):
            pc_index.upsert(vectors=vectors_to_upsert[i:i+50])
        print(f'successfully indexed {len(vectors_to_upsert)} chunks into pinecone') 

    except Exception as e:
        print(f'Pinecone indexing notice: {e}')

else:
        print('Pinecone api key not found  in environment')

            




    

successfully indexed 110 chunks into pinecone


In [12]:
with open(os.path.join(VECTOR_DB_DIR, 'chunks_store.json'), 'w', encoding='utf') as f:
    json.dump([{'page_content': c.page_content, 'metadata': c.metadata} for c in chunks], f, indent=2)

In [13]:
import chromadb, os
client = chromadb.PersistentClient(path='vector_db/chroma')
coll = client.get_or_create_collection("clouddesk_support_kb")
print("Count:", coll.count())
sample = coll.get(limit=5, include=['metadatas'])
for m in sample['metadatas']:
    print("→", m.get('title'))

Count: 110
→ Authentication
→ Authentication
→ Authentication
→ Authentication
→ Authentication
